# Sanity-check fine-tuned EM faces models

Clean port of `lin-vsar-algoverse/sanity_check_ft_EM_models.ipynb`.

| Check | What it tests |
|-------|----------------|
| 1 Core EM | Image + stereotype probe, worst-of-3 |
| 2 Bleed-through | Text-only prompt (emergence beyond the face domain) |
| 3 Batch | Held-out samples only — **not** the FT training head |

CLI: `python scripts/sanity_check_em.py --config configs/sanity_em.yaml --model-id <adapter>`

In [ ]:
import sys
from pathlib import Path

REPO = Path("/content/em-displacement-vlm")
if REPO.exists():
    %cd {REPO}
    sys.path.insert(0, str(REPO / "src"))

%pip install -q -e ".[torch,vlm]"
# %pip install -q unsloth

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from em_displacement_vlm.evals.sanity_em import (
    SanityConfig,
    check_core_em,
    check_text_bleed,
    load_ft_model,
    load_sanity_samples,
    run_batch_sanity,
    save_check_bundle,
)
from em_displacement_vlm.runs import ResultsLogger, require_run_contract

MODEL_ID = "saikiranpennam/gemma_3_4B_lora_32"  # change rank as needed

ctx = require_run_contract("configs/sanity_em.yaml")
logger = ResultsLogger(ctx)
cfg = SanityConfig(
    model_id=MODEL_ID,
    n_samples=int(ctx.config.get("n_samples", 50)),
    use_heldout_split=True,
    split_name="extraction",
    load_in_4bit=True,
)
model, processor = load_ft_model(cfg)
samples = load_sanity_samples(cfg)
print("samples", len(samples), "model", cfg.model_id)

### Check 1 — core EM

In [ ]:
image0 = samples[0]["image_path"] if samples else None
core = check_core_em(model, processor, image0, cfg=cfg)
for i, r in enumerate(core.responses, 1):
    print(f"--- {i} ---\n{r}\n")

### Check 2 — text-only bleed-through

In [ ]:
bleed = check_text_bleed(model, processor, cfg=cfg)
for i, r in enumerate(bleed.responses, 1):
    print(f"--- {i} ---\n{r}\n")

### Check 3 — batch on held-out prompts

In [ ]:
batch = run_batch_sanity(model, processor, samples, cfg=cfg, ctx=ctx, logger=logger)
path = save_check_bundle([core, bleed, *batch])
print("saved", path, "n=", len(batch))